In [ ]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import re, os, time, json, ast
from dotenv import load_dotenv

load_dotenv(".env")
API_KEY = os.getenv("GEMINI_API_KEY")
print(API_KEY)

client = genai.Client(
    api_key=API_KEY
)

In [ ]:
df_final = pd.read_csv("3000_data_gabungan.csv")
df_o_notna = df_final[df_final['O'].notna()].copy()

chunk_size = 50
chunks = [df_o_notna.iloc[i:i + chunk_size] for i in range(0, len(df_o_notna), chunk_size)]

processed_chunks = []

In [ ]:
for xx in range(len(chunks)):
    df = chunks[xx].copy()
    cols = ['nadi', 'suhu','pernapasan','SPO2','tinggiBadan', 'beratBadan','tekananDarah']
    extracted_rows = []

    for index, row in df.iterrows():
        time.sleep(5)

        extracted = {"index": row["index"]}
        print(index)

        raw_text = row['O']
        if pd.isna(raw_text) or not isinstance(raw_text, str) or raw_text.strip() == "":
            continue

        # Create a specific prompt for each column
        prompt_text = f"""
            Ekstrak 'nadi', 'suhu','pernapasan','SPO2','tinggiBadan', 'beratBadan','tekananDarah' dari teks di bawah ini.
            Berikan hasilnya dalam format JSON tanpa tambahan teks atau penjelasan.

            Teks:
            \"{raw_text.strip()}\"
            """

        try:
            model = "gemini-2.0-flash"
            contents = [
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=prompt_text),
                    ],
                ),
            ]
            generate_content_config = types.GenerateContentConfig(
                response_mime_type="text/plain",
            )

            response_text=""
            for chunk in client.models.generate_content_stream(
                model=model,
                contents=contents,
                config=generate_content_config,
            ): response_text += chunk.text

            clean_text = re.sub(r"```json|```", "", response_text).strip()
            xdata = json.loads(clean_text)
            extracted[f"O_extracted"] = xdata

        except Exception as e:
            print(f"Row {index} - Error processing column :", e)
            extracted[f"O_extracted"] = None

        extracted_rows.append(extracted)

        dff = pd.DataFrame(extracted_rows)
        dff.to_csv(f"./data_extract/extract{xx}.csv")


In [ ]:
gabungan_df = pd.DataFrame([])

for i in range(20):
    inputdf = pd.read_csv(f'./data_extract/extract{i}.csv')
    gabungan_df = pd.concat([gabungan_df, inputdf], ignore_index=True)

gabungan_df.tail()

def safe_eval(x):
    if pd.isna(x):
        return {}
    try:
        return ast.literal_eval(x)
    except Exception:
        return {}

df_parsed = gabungan_df['O_extracted'].apply(safe_eval).apply(pd.Series)
gabungan_df = pd.concat([gabungan_df, df_parsed], axis=1)

columns_to_replace = ['nadi', 'suhu', 'pernapasan', 'SPO2', 'tinggiBadan', 'beratBadan', 'tekananDarah']
df_final.update(gabungan_df[columns_to_replace])

In [ ]:
chunk_size = 50
chunks = [df_final.iloc[i:i + chunk_size] for i in range(0, len(df_final), chunk_size)]

processed_chunks = []

In [ ]:
for xx in range(len(chunks)):
    df = chunks[xx].copy()
    extracted_rows = []

    for index, row in df.iterrows():
        time.sleep(5)

        extracted = {"index": row["index"]}
        print(index)

        raw_text = row['S']
        if pd.isna(raw_text) or not isinstance(raw_text, str) or raw_text.strip() == "":
            raw_text = row['riwayatPenyakit']
            if pd.isna(raw_text) or not isinstance(raw_text, str) or raw_text.strip() == "":
                continue

        prompt_text = f"""
            Ekstrak semua keluhan utama secara eksplisit dari teks berikut.
            - Hanya ekstrak yang jelas disebutkan dalam teks.
            - Gunakan format JSON dengan satu key: "keluhan_utama".
            - Nilai dari "keluhan_utama" adalah array berisi string keluhan.
            - Jika tidak ada keluhan, isi array dengan satu item: "Tidak ada keluhan".
            - Jangan tambahkan teks atau penjelasan di luar JSON.

            Teks:
            \"{raw_text.strip()}\"
            """

        try:
            model = "gemini-2.0-flash"
            contents = [
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=prompt_text),
                    ],
                ),
            ]
            generate_content_config = types.GenerateContentConfig(
                response_mime_type="text/plain",
            )

            response_text=""
            for chunk in client.models.generate_content_stream(
                model=model,
                contents=contents,
                config=generate_content_config,
            ): response_text += chunk.text

            clean_text = re.sub(r"```json|```", "", response_text).strip()
            xdata = json.loads(clean_text)
            extracted[f"keluhan_extracted"] = xdata

        except Exception as e:
            print(f"Row {index} - Error processing column :", e)
            extracted[f"keluhan_extracted"] = None

        extracted_rows.append(extracted)

        dff = pd.DataFrame(extracted_rows)
        dff.to_csv(f"./keluhan_extract_done/extract{xx}.csv")

In [ ]:
gabungan_df2 = pd.DataFrame([])

for i in range(60):
    inputdf = pd.read_csv(f'./keluhan_extract_done/extract{i}.csv')
    gabungan_df2 = pd.concat([gabungan_df2, inputdf], ignore_index=True)

merged_df = pd.merge(df_final, gabungan_df2, on='index', how='left')

merged_df["keluhan_extracted"] = merged_df["keluhan_extracted"].fillna("{'keluhan_utama': ['sakit']}")
merged_df['pengobatan'] = merged_df['P'].combine_first(merged_df['hasilIntruksi'])

pattern = r"^Unnamed:"
merged_df = merged_df[[col for col in merged_df.columns if not re.match(pattern, col)]]

def extract_keluhan_string(x):
    if pd.isna(x):
        return None
    try:
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(parsed, dict) and 'keluhan_utama' in parsed:
            return ', '.join(parsed['keluhan_utama'])
    except:
        return None

merged_df['keluhan_utama_str'] = merged_df['keluhan_extracted'].apply(extract_keluhan_string)

In [ ]:
merged_df['beratBadan'] = (
    merged_df['beratBadan']
    .astype(str)
    .str.replace(r'[^0-9.,]', '', regex=True)  
    .str.replace(',', '.', regex=False)       
)
merged_df['beratBadan'] = pd.to_numeric(merged_df['beratBadan'], errors='coerce')

merged_df['tinggiBadan'] = (
    merged_df['tinggiBadan']
    .astype(str)
    .str.replace(r'[^0-9.,]', '', regex=True)
    .str.replace(',', '.', regex=False)
)
merged_df['tinggiBadan'] = pd.to_numeric(merged_df['tinggiBadan'], errors='coerce')

merged_df['tekananDarah'] = (
    merged_df['tekananDarah']
    .astype(str)
    .str.replace("\\", "/", regex=False)       
    .str.replace(r'[^0-9/]', '', regex=True)   
)
split_bp = merged_df['tekananDarah'].str.split('/', n=1, expand=True)
merged_df['tekanan_sistolik'] = pd.to_numeric(split_bp[0], errors='coerce')
merged_df['tekanan_diastolik'] = pd.to_numeric(split_bp[1], errors='coerce')


merged_df['suhu'] = (
    merged_df['suhu']
    .astype(str)
    .str.replace(',', '.', regex=False)       
    .str.replace(r'[^0-9.]', '', regex=True)   
)
merged_df['suhu'] = pd.to_numeric(merged_df['suhu'], errors='coerce')

merged_df['nadi'] = (
    merged_df['nadi']
    .astype(str)
    .str.replace(r'[^0-9]', '', regex=True)
)
merged_df['nadi'] = pd.to_numeric(merged_df['nadi'], errors='coerce')

merged_df['pernapasan'] = (
    merged_df['pernapasan']
    .astype(str)
    .str.replace(r'[^0-9]', '', regex=True)
)
merged_df['pernapasan'] = pd.to_numeric(merged_df['pernapasan'], errors='coerce')

merged_df.info()

In [ ]:
pattern = r"^Unnamed:"
merged_df = merged_df[[col for col in merged_df.columns if not re.match(pattern, col)]]

merged_df = merged_df.rename(columns={
    'O' : 'detailPemeriksaan',
    'S' : 'detailKeluhan',
    'P'  : 'detailPengobatan',
})

merged_df.to_csv("final_extracted_3000data.csv")